# Locally Adapted Conformal Prediction

In [11]:
MC_simulation_samples = 1000
n_models_ensemble = 1000
alpha = 0.05

dataset_name_list = ["train", "test", "calib"]

valor_max_list = [10]*3
valor_min_list = [0]*3
num_amostras_list = [100, 1000, 0]

erro_sistematico_x_list = [0.0]*3
erro_aleatorio_x_list = [0.2]*3
dist_erro_x_list = ["uniforme"]*3

erro_sistematico_y_list = [0]*3
erro_aleatorio_y_list = [0.1]*3
dist_erro_y_list = ["uniforme"]*3

gerar_mc_list = [True, False, False]

## Ambiente Básico

#### Imports

In [2]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import time
import optuna

import tqdm

from modelo_scr.utils import *

from modelo_scr.modelo_pacheco import *
from modelo_scr.nn_ensamble import *
from modelo_scr.rf_ensamble import *
from modelo_scr.conformal_prediction import *

from sklearn.metrics import mean_absolute_error

#### Modelo do sistema simuado

In [3]:
def modelo(x):
    return 10*x**2

#### Gera os dataset

In [4]:
dataset = {}

for i in range(len(dataset_name_list)):
    print(f"Gerando dataset {dataset_name_list[i]}:")
    dataset[dataset_name_list[i]] = Dataset(modelo, 
                                            valor_max_list[i], 
                                            valor_min_list[i], 
                                            num_amostras_list[i],
                                            erro_sistematico_x_list[i], 
                                            erro_aleatorio_x_list[i], 
                                            dist_erro_x_list[i], 
                                            erro_sistematico_y_list[i], 
                                            erro_aleatorio_y_list[i], 
                                            dist_erro_y_list[i])
    if gerar_mc_list[i]:
        dataset[dataset_name_list[i]].gerar_monte_carlo(MC_simulation_samples)


Gerando dataset train:
Shape X_measured: (100,)
Shape y_measured: (100,)
Shape X_measured_mc: (100000,)
Shape y_measured_mc: (100000,)
Gerando dataset test:
Shape X_measured: (1000,)
Shape y_measured: (1000,)
Gerando dataset calib:
Shape X_measured: (0,)
Shape y_measured: (0,)


## Treinamento da NN Ensable

In [5]:
# Definição do modelo
def model_fn():
    return MLP(input_dim=1, hidden_layers=[72]*2, activation="relu", dropout=0.001)

# Configuração
config = TrainerConfig(epochs=1000, lr=1e-3)

Usando dispositivo: cuda


### Modelo treinado com train dataset
Utiliza os hiperparâmetros definidos na otimização do modelo do pacheco uma vez que a arquitetura é a mesma, só que sem o ensable de incerteza.

In [6]:
modelo_nn = NNEnsambleModel(model_fn,
                config,
                n_models=n_models_ensemble,
                verbose=True)

modelo_nn.fit(dataset["train"].X_measured.reshape((-1, 1)).numpy(), 
              dataset["train"].y_measured.numpy())

Treinando ensemble de inferência do valor...


100%|██████████| 1000/1000 [27:28<00:00,  1.65s/it]

Treinamento concluído em 1648.14s


In [14]:
modelo_nn.save("modelo_nn_ensamble_tds_2.pt")

Modelo salvo em: modelo_nn_ensamble_tds_2.pt


### Modelo treinado com MC do train dataset

In [8]:
modelo_nn_tmc = NNEnsambleModel(model_fn,
                config,
                n_models=n_models_ensemble,
                verbose=True)

modelo_nn_tmc.fit(dataset["train"].X_measured_mc.reshape((-1, 1)).numpy(), 
                  dataset["train"].y_measured_mc.numpy(), 
                  ds_size=dataset["train"].num_amostras)

Treinando ensemble de inferência do valor...


  0%|          | 1/1000 [00:00<14:27,  1.15it/s]

100%|██████████| 1000/1000 [30:10<00:00,  1.81s/it]

Treinamento concluído em 1810.92s


In [15]:
modelo_nn_tmc.save("modelo_nn_ensamble_tmc2.pt")

Modelo salvo em: modelo_nn_ensamble_tmc2.pt


### Modelo Pacheco 

In [10]:
modelo = UQModel(model_fn,
                config,
                n_models=n_models_ensemble,
                mcs_samples=MC_simulation_samples,
                input_std=dataset["train"].erro_aleatorio_x,
                u_M=dataset["train"].erro_aleatorio_y,
                k=2,
                u_M_abs=False,
                verbose=True)

modelo.fit(dataset["train"].X_measured.reshape((-1, 1)).numpy(),
           dataset["train"].y_measured.numpy())

Treinando ensemble de inferência do valor...


100%|██████████| 1000/1000 [31:57<00:00,  1.92s/it]


Treinando ensemble de incerteza...


100%|██████████| 1000/1000 [29:30<00:00,  1.77s/it]

Treinamento concluído em 3688.29s


In [16]:
modelo.save("modelo_pacheco_tds2.pt")

Modelo salvo em: modelo_pacheco_tds2.pt


## Modelos de Estimativa

In [5]:
model_dict = {"rf": RFEnsambleModel(dataset["train"].X_measured,
                                    dataset["train"].y_measured,
                                    n_models_ensemble=n_models_ensemble,
                                    num_amostras_treino=dataset["train"].num_amostras),
              "nn": NNEnsambleModel.load("modelo_nn_ensamble_tds.pt"),
              "rf_tmc": RFEnsambleModel(dataset["train"].X_measured_mc,
                                        dataset["train"].y_measured_mc,
                                        n_models_ensemble=n_models_ensemble,
                                        num_amostras_treino=dataset["train"].num_amostras),
              "nn_tmc": NNEnsambleModel.load("modelo_nn_ensamble_tmc.pt"),
              "pacheco": UQModel.load("modelo_pacheco_tds.pt")}

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    0.5s


Usando dispositivo: cuda


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 799 tasks      | elapsed:    0.8s


Usando dispositivo: cuda
Usando dispositivo: cuda


In [8]:
cp_dict = {}
uncertainty_dict = {}
for name, model in model_dict.items():
    if name != "pacheco":
        print(f"Avaliando modelo {name}:")
        cp_dict[name] = CPCalibration(model,
                                    dataset_calib=dataset["train"],
                                    mc=not("tmc" in name),
                                    alpha=alpha)
        uncertainty_dict[name] = UncertaintyEvaluator(model, 
                                                    cp_dict[name].cp_dict,
                                                    dataset["test"], 
                                                    alpha=alpha,
                                                    num_mc_simulations=MC_simulation_samples, 
                                                    verbose=True)
        print("="*50)
    else:
        print(f"Avaliando modelo {name}:")
        uncertainty_dict[name] = ModelUncertaintyEvaluator(model, 
                                                            dataset["test"], 
                                                            alpha=alpha)
        print("="*50)
    

Avaliando modelo rf:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |      69.8545%|      149.5616
absolute    |      96.9854%|      542.2321
normalized  |      98.8565%|      469.2450
monte_carlo |      98.3368%|      313.9473
Avaliando modelo nn:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |       0.0000%|        0.0001
absolute    |      99.8960%|      531.9908
normalized  |     100.0000%|      381.6151
monte_carlo |      91.6840%|      213.4963
Avaliando modelo rf_tmc:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model       |      98.2328%|      329.8579
absolute    |      95.2183%|      394.3628
normalized  |      95.8420%|      247.2600
monte_carlo |     100.0000%|      385.4908
Avaliando modelo nn_tmc:
Model       |   Coverage   |   Average Width
--------------------------------------------------
model   

In [10]:
for name_model in uncertainty_dict.keys():
    for name in uncertainty_dict[name_model].y_pred.keys():
        print(f'Gráfico do modelo {name_model} com {name}:')
        uncertainty_dict[name_model].graph_with_uncertainty(name, plot_true=True, plot_measured=False)

Gráfico do modelo rf com model:


Gráfico do modelo rf com absolute:


Gráfico do modelo rf com normalized:


Gráfico do modelo rf com monte_carlo:


Gráfico do modelo nn com model:


...

Gráfico do modelo nn_tmc com model:


Gráfico do modelo nn_tmc com absolute:


Gráfico do modelo nn_tmc com normalized:


Gráfico do modelo nn_tmc com monte_carlo:


Gráfico do modelo pacheco com model:
